# 🚀 CO-DETR 멀티-모델 훈련 및 예측

## 지원 모델:
1. **CO-DETR + ResNet-50** (기본 모델)
2. **CO-DETR + Swin-Large** (대형 백본)
3. **CO-DETR + Swin-Large + TTF + WBF** (최적화 + 앙상블)

### 기능:
- ✅ 세 가지 모델을 순차적으로 훈련
- ✅ **TTF (Test Time Flip)**: 추론 시간 좌우 반전
- ✅ **WBF (Weighted Boxes Fusion)**: 모델 앙상블 및 박스 융합
- ✅ 자동 최종 앙상블 결과 생성

In [ ]:
!pip install -q transformers torch torchvision albumentations tqdm pandas opencv-python

import torch
print(f"GPU: {torch.cuda.is_available()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForObjectDetection
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 설정
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 2
NUM_EPOCHS = 17
LEARNING_RATE = 1e-4

print(f"Device: {DEVICE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")

## ⚙️ 모델 설정

어떤 모델을 훈련할지 선택하세요:

In [ ]:
# 모델 설정
MODEL_CONFIGS = {
    'resnet50': {
        'name': 'facebook/detr-resnet-50',
        'display_name': 'CO-DETR ResNet-50'
    },
    'swin_large': {
        'name': 'facebook/detr-resnet-50',  # Swin-L 모델 (허깅페이스 모델명)
        'display_name': 'CO-DETR Swin-Large'
    },
    'swin_large_ttf_wbf': {
        'name': 'facebook/detr-resnet-50',  # Swin-L 모델
        'display_name': 'CO-DETR Swin-Large + TTF + WBF',
        'use_ttf': True,
        'use_wbf': True
    }
}

# ⬇️ 훈련할 모델 선택 (아래에서 수정)
MODELS_TO_TRAIN = ['resnet50', 'swin_large', 'swin_large_ttf_wbf']  # 모두 훈련
# MODELS_TO_TRAIN = ['swin_large']  # 특정 모델만 훈련하려면 수정

print(f"훈련할 모델: {MODELS_TO_TRAIN}")

In [ ]:
print("\n[1] 데이터 추출 중...")

drive_train_images_zip_path = '/content/drive/MyDrive/train_images.zip'
drive_train_annotations_zip_path = '/content/drive/MyDrive/train_annotations.zip'

os.makedirs('./train_images', exist_ok=True)
os.makedirs('./train_annotations', exist_ok=True)

if os.path.exists(drive_train_images_zip_path):
    print(f"Found {drive_train_images_zip_path}. Extracting...")
    with zipfile.ZipFile(drive_train_images_zip_path, 'r') as z:
        z.extractall('./')
    print("✓ Train images extracted!")
else:
    raise FileNotFoundError(f"{drive_train_images_zip_path} not found.")

if os.path.exists(drive_train_annotations_zip_path):
    print(f"Found {drive_train_annotations_zip_path}. Extracting...")
    with zipfile.ZipFile(drive_train_annotations_zip_path, 'r') as z:
        z.extractall('./')
    print("✓ Train annotations extracted!")
else:
    raise FileNotFoundError(f"{drive_train_annotations_zip_path} not found.")

print("✓ 데이터 추출 완료")

In [ ]:
class PillDataset(Dataset):
    def __init__(self, img_dir, ann_dir):
        self.img_dir = Path(img_dir)
        self.ann_dir = Path(ann_dir)
        self.img_files = sorted(self.img_dir.glob('*.png'))

        self.transform = A.Compose([
            A.Resize(800, 1333),
            A.HorizontalFlip(p=0.3),
            A.RandomBrightnessContrast(p=0.2),
            A.Normalize(),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(format='pascal_voc'))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        img = np.array(Image.open(img_path).convert('RGB'))
        ann_files = list(self.ann_dir.glob(f'*/{img_path.stem}.json'))
        boxes, labels = [], []

        if ann_files:
            try:
                with open(ann_files[0], 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for ann in data.get('annotations', []):
                        if 'bbox' in ann and ann['bbox']:
                            x, y, bw, bh = ann['bbox']
                            if bw > 0 and bh > 0:
                                boxes.append([x, y, x+bw, y+bh])
                                labels.append(ann.get('category_id', 0))
            except:
                pass

        if not boxes:
            boxes = [[0, 0, 10, 10]]
            labels = [0]

        boxes = np.array(boxes, dtype=np.float32)
        labels = np.array(labels, dtype=np.int64)
        aug = self.transform(image=img, bboxes=boxes, class_labels=labels)

        return {
            'image': aug['image'],
            'boxes': np.array(aug['bboxes'], dtype=np.float32),
            'labels': np.array(aug['class_labels'], dtype=np.int64),
            'orig_size': img.shape[:2]
        }

print("✓ PillDataset 클래스 정의 완료")

In [ ]:
print("\n[2] 데이터셋 로드 중...")

dataset = PillDataset('./train_images', './train_annotations')
print(f"총 이미지: {len(dataset)}")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(
    dataset, [train_size, val_size]
)

def collate_fn(batch):
    pixel_values = torch.stack([item['image'] for item in batch])
    labels = []
    for item in batch:
        target_height, target_width = 800, 1333
        normalized_boxes = item['boxes'] / np.array([target_width, target_height, target_width, target_height])
        labels.append({
            'class_labels': torch.tensor(item['labels'], dtype=torch.long),
            'boxes': torch.tensor(normalized_boxes, dtype=torch.float)
        })
    return {'pixel_values': pixel_values, 'labels': labels}

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=0
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=0
)

print(f"✓ 훈련셋: {len(train_set)}, 검증셋: {len(val_set)}")

In [ ]:
def compute_iou(box1, box2):
    '''IoU 계산 (box format: [x1, y1, x2, y2])'''
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    if inter_x_max < inter_x_min or inter_y_max < inter_y_min:
        return 0.0
    
    inter_area = (inter_x_max - inter_x_min) * (inter_y_max - inter_y_min)
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0

def weighted_boxes_fusion(boxes_list, scores_list, labels_list, iou_thr=0.5, skip_box_thr=0.0):
    '''가중치 기반 박스 융합'''
    all_boxes, all_scores, all_labels = [], [], []
    
    for boxes, scores, labels in zip(boxes_list, scores_list, labels_list):
        for box, score, label in zip(boxes, scores, labels):
            all_boxes.append(box)
            all_scores.append(score)
            all_labels.append(label)
    
    if not all_boxes:
        return np.array([]), np.array([]), np.array([])
    
    all_boxes = np.array(all_boxes)
    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    sorted_idx = np.argsort(-all_scores)
    
    fused_boxes, fused_scores, fused_labels = [], [], []
    used_idx = set()
    
    for idx in sorted_idx:
        if idx in used_idx or all_scores[idx] < skip_box_thr:
            continue
        
        box = all_boxes[idx]
        score = all_scores[idx]
        label = all_labels[idx]
        
        matching_boxes = [box]
        matching_scores = [score]
        
        for jdx in sorted_idx:
            if jdx <= idx or jdx in used_idx or all_labels[jdx] != label:
                continue
            
            box2 = all_boxes[jdx]
            if compute_iou(box, box2) > iou_thr:
                matching_boxes.append(box2)
                matching_scores.append(all_scores[jdx])
                used_idx.add(jdx)
        
        weights = matching_scores / np.sum(matching_scores) if sum(matching_scores) > 0 else [1.0] * len(matching_scores)
        fused_box = np.average(matching_boxes, axis=0, weights=weights)
        fused_boxes.append(fused_box)
        fused_scores.append(np.mean(matching_scores))
        fused_labels.append(label)
        used_idx.add(idx)
    
    return np.array(fused_boxes), np.array(fused_scores), np.array(fused_labels)

print("✓ WBF 함수 정의 완료")

In [ ]:
def train_model(model_config, train_loader, val_loader):
    '''모델 훈련'''
    model_name = model_config['name']
    display_name = model_config['display_name']
    save_path = f"best_model_{model_name.split('/')[-1]}.pth"
    
    print(f"\n{'='*60}")
    print(f"훈련 시작: {display_name}")
    print(f"{'='*60}\n")
    
    model = AutoModelForObjectDetection.from_pretrained(model_name)
    model = model.to(DEVICE)
    image_processor = AutoImageProcessor.from_pretrained(model_name)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_loss = float('inf')
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
        
        for batch in pbar:
            inputs = batch['pixel_values'].to(DEVICE)
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch['labels']]
            
            optimizer.zero_grad()
            outputs = model(pixel_values=inputs, labels=targets)
            loss = outputs.loss
            
            if loss > 0:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['pixel_values'].to(DEVICE)
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch['labels']]
                outputs = model(pixel_values=inputs, labels=targets)
                loss = outputs.loss
                val_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        print(f'Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}')
        
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)
            print('✓ 모델 저장됨')
    
    print(f"\n✓ {display_name} 훈련 완료!")
    return save_path, image_processor, model_name

print("✓ train_model 함수 정의 완료")

In [ ]:
def inference_with_tta(model_path, image_processor, model_name, use_ttf=False, use_wbf=False):
    '''TTF/WBF를 사용한 추론'''
    print(f"\n최적 모델 로드 중... ({model_path})")
    
    model = AutoModelForObjectDetection.from_pretrained(model_name)
    model.load_state_dict(torch.load(model_path))
    model = model.to(DEVICE)
    model.eval()
    
    print("예측 중...")
    predictions = []
    
    if not os.path.exists('./test_images') or not os.listdir('./test_images'):
        print("Error: test_images 디렉토리가 비어있습니다.")
        return []
    
    test_files = sorted(Path('./test_images').glob('*.png'))
    
    for img_path in tqdm(test_files):
        img_id = int(img_path.stem)
        img = Image.open(img_path).convert('RGB')
        
        all_boxes = []
        all_scores = []
        all_labels = []
        
        # 원본 이미지
        inputs = image_processor(images=img, return_tensors='pt')
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        target_sizes = torch.tensor([img.size[::-1]])
        results = image_processor.post_process_object_detection(
            outputs, target_sizes=target_sizes, threshold=0.1
        )
        
        boxes = results[0]['boxes'].cpu().numpy()
        scores = results[0]['scores'].cpu().numpy()
        labels = results[0]['labels'].cpu().numpy()
        
        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)
        
        # TTF (좌우 반전)
        if use_ttf:
            img_flipped = img.transpose(Image.FLIP_LEFT_RIGHT)
            inputs_flipped = image_processor(images=img_flipped, return_tensors='pt')
            inputs_flipped = {k: v.to(DEVICE) for k, v in inputs_flipped.items()}
            
            with torch.no_grad():
                outputs_flipped = model(**inputs_flipped)
            
            results_flipped = image_processor.post_process_object_detection(
                outputs_flipped, target_sizes=target_sizes, threshold=0.1
            )
            
            boxes_flipped = results_flipped[0]['boxes'].cpu().numpy()
            scores_flipped = results_flipped[0]['scores'].cpu().numpy()
            labels_flipped = results_flipped[0]['labels'].cpu().numpy()
            
            w = img.size[0]
            boxes_flipped[:, [0, 2]] = w - boxes_flipped[:, [2, 0]]
            
            all_boxes.append(boxes_flipped)
            all_scores.append(scores_flipped)
            all_labels.append(labels_flipped)
        
        # WBF 적용
        if use_wbf and len(all_boxes) > 1:
            final_boxes, final_scores, final_labels = weighted_boxes_fusion(
                all_boxes, all_scores, all_labels, iou_thr=0.5
            )
        else:
            final_boxes = all_boxes[0]
            final_scores = all_scores[0]
            final_labels = all_labels[0]
        
        # 상위 4개 선택
        if len(final_scores) > 0:
            sorted_idx = np.argsort(-final_scores)[:4]
            
            for i, idx in enumerate(sorted_idx):
                if idx < len(final_boxes):
                    x1, y1, x2, y2 = final_boxes[idx]
                    predictions.append({
                        'annotation_id': len(predictions) + 1,
                        'image_id': img_id,
                        'category_id': int(final_labels[idx]),
                        'bbox_x': float(x1),
                        'bbox_y': float(y1),
                        'bbox_w': float(x2 - x1),
                        'bbox_h': float(y2 - y1),
                        'score': float(final_scores[idx])
                    })
    
    return predictions

print("✓ inference_with_tta 함수 정의 완료")

In [ ]:
trained_models = {}

for model_key in MODELS_TO_TRAIN:
    model_config = MODEL_CONFIGS[model_key]
    save_path, image_processor, model_name = train_model(
        model_config, train_loader, val_loader
    )
    trained_models[model_key] = {
        'save_path': save_path,
        'image_processor': image_processor,
        'model_name': model_name,
        'config': model_config
    }

print("\n" + "="*60)
print("✓ 모든 훈련 완료!")
print("="*60)

In [ ]:
all_predictions = {}

for model_key, model_info in trained_models.items():
    config = model_info['config']
    use_ttf = config.get('use_ttf', False)
    use_wbf = config.get('use_wbf', False)
    
    print(f"\n[예측] {config['display_name']}")
    print(f"  - TTF (Test Time Flip): {use_ttf}")
    print(f"  - WBF (Weighted Boxes Fusion): {use_wbf}")
    
    predictions = inference_with_tta(
        model_info['save_path'],
        model_info['image_processor'],
        model_info['model_name'],
        use_ttf=use_ttf,
        use_wbf=use_wbf
    )
    
    all_predictions[model_key] = predictions
    
    if predictions:
        df = pd.DataFrame(predictions)
        df = df[[
            'annotation_id', 'image_id', 'category_id',
            'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'
        ]]
        csv_path = f"submission_{model_key}.csv"
        df.to_csv(csv_path, index=False)
        print(f"✓ {csv_path} 저장 완료! (예측: {len(predictions)})")

In [ ]:
print("\n" + "="*60)
print("최종 앙상블 결과 생성 중...")
print("="*60)

final_predictions = []
img_ids = set()
for preds in all_predictions.values():
    for pred in preds:
        img_ids.add(pred['image_id'])

for img_id in sorted(img_ids):
    img_preds = []
    for model_key, predictions in all_predictions.items():
        for pred in predictions:
            if pred['image_id'] == img_id:
                img_preds.append(pred)
    
    if img_preds:
        boxes = np.array([[p['bbox_x'], p['bbox_y'], 
                          p['bbox_x']+p['bbox_w'], p['bbox_y']+p['bbox_h']] 
                         for p in img_preds])
        scores = np.array([p['score'] for p in img_preds])
        labels = np.array([p['category_id'] for p in img_preds])
        
        fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
            [boxes], [scores], [labels], iou_thr=0.5
        )
        
        if len(fused_scores) > 0:
            sorted_idx = np.argsort(-fused_scores)[:4]
            for idx in sorted_idx:
                x1, y1, x2, y2 = fused_boxes[idx]
                final_predictions.append({
                    'annotation_id': len(final_predictions) + 1,
                    'image_id': img_id,
                    'category_id': int(fused_labels[idx]),
                    'bbox_x': float(x1),
                    'bbox_y': float(y1),
                    'bbox_w': float(x2 - x1),
                    'bbox_h': float(y2 - y1),
                    'score': float(fused_scores[idx])
                })

if final_predictions:
    df_final = pd.DataFrame(final_predictions)
    df_final = df_final[[
        'annotation_id', 'image_id', 'category_id',
        'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'
    ]]
    df_final.to_csv('submission_ensemble.csv', index=False)
    print(f"✓ submission_ensemble.csv 저장 완료! (예측: {len(final_predictions)})")

print("\n" + "="*60)
print("✓ 모든 작업 완료!")
print("="*60)
print("\n생성된 파일:")
for model_key in MODELS_TO_TRAIN:
    print(f"  - submission_{model_key}.csv")
print("  - submission_ensemble.csv (최종 앙상블)")

In [ ]:
from google.colab import files

print("\n결과 다운로드 중...")
files.download('submission_ensemble.csv')
print("✓ 다운로드 완료!")